In [ ]:
# Import necessary libraries
import os
import sys
# Add the utils folder to the system path so we can import custom modules
sys.path.append(os.path.abspath(os.path.join("../..")))
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
import pandas as pd
from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data, feature_names

# Set the plotting style for better-looking visualizations
plt.style.available
plt.style.use('seaborn-v0_8')

In [ ]:
print("Loading train data...")
train_dataloader, val_dataloader, pipeline = load_data(
    train_path = "../../datasets_challenge/train",  # Path to training data
    train_ratio = 0.99,  # Use 99% for training, 1% for validation
    batchsize = 64,  # Number of samples per batch
    # columns_to_standardize = feature_names,  # Option to specify which columns to standardize
    columns_to_drop = ["sideSlip", "vyCG", "time", "longitude","latitude"],  # Remove these from features
    sequence_length = 20,  # Number of time steps in each sequence (t-19 to t0)
    threshold = True,  # Enable velocity thresholding
    threshold_value = 3.0,  # Only keep samples where vehicle speed > 3 m/s
    threshold_column = "vxCG",  # Column to apply threshold on (longitudinal velocity)
    target = "sideSlip",  # The variable we want to predict
    scaling = True,  # Apply standardization to features
    seed = 42,  # Random seed for train/val split
)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random

class RNN(nn.Module):
    def __init__(self, input_size, output_size=1, rnn_units=28, fc_units=28, 
                 activation_fct=nn.ReLU(), dropout=[0.0, 0, 0], rnn_nonlinearity='relu'):
        super(RNN, self).__init__()

        self.rnn_layer = nn.RNN(
            input_size=input_size,
            hidden_size=rnn_units,
            num_layers=1,
            nonlinearity=rnn_nonlinearity,
            batch_first=True
        )

        self.fc = nn.Linear(rnn_units, fc_units)
        self.output_layer = nn.Linear(fc_units, output_size)

        self.dropout1 = nn.Dropout(dropout[0])
        self.dropout2 = nn.Dropout(dropout[1])
        self.dropout3 = nn.Dropout(dropout[2])

        self.activation_fct = activation_fct

    def forward(self, x):
        x = self.dropout1(x)
        x, _ = self.rnn_layer(x)
        x = x[:, -1, :]
        x = self.dropout2(x)
        x = self.activation_fct(self.fc(x))
        x = self.dropout3(x)
        x = self.output_layer(x)
        return x


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [ ]:
ensemble_configs = [
    {"seed": 42,   "lr": 3e-4, "rnn_units": 48, "fc_units": 48},
    {"seed": 123,  "lr": 4e-4, "rnn_units": 56, "fc_units": 56},
    {"seed": 777,  "lr": 2e-4, "rnn_units": 64, "fc_units": 64},
    {"seed": 2025, "lr": 5e-4, "rnn_units": 40, "fc_units": 40},
    {"seed": 99,   "lr": 6e-4, "rnn_units": 52, "fc_units": 52},
]


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_single_rnn(model_id, config, train_loader, val_loader, num_epochs=40):

    print(f"\n===== Training Model {model_id} with seed={config['seed']} =====")
    seed_everything(config["seed"])

    model = RNN(
        input_size=train_loader.dataset[0][0].shape[1],
        rnn_units=config["rnn_units"],
        fc_units=config["fc_units"],
        dropout=[0.0,0.0,0.0]
    ).to(device)

    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])

    best_loss = float("inf")

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            
            optimizer.zero_grad()
            preds = model(X)
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)

        # Validation
        model.eval()
        running_val = 0.0
        with torch.no_grad():
            for Xv, yv in val_loader:
                Xv, yv = Xv.to(device), yv.to(device)
                pred_val = model(Xv)
                val_loss = criterion(pred_val, yv)
                running_val += val_loss.item()

        avg_val_loss = running_val / len(val_loader)

        print(f"Model {model_id} | Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")

        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            torch.save(model.state_dict(), f"ensemble_model_{model_id}.pth")

    print(f"Saved best ensemble_model_{model_id}.pth")


In [ ]:
checkpoint_paths = []

for i, cfg in enumerate(ensemble_configs, start=1):
    train_single_rnn(i, cfg, train_dataloader, val_dataloader)
    checkpoint_paths.append(f"ensemble_model_{i}.pth")


In [ ]:
def ensemble_predict(test_loader, checkpoint_paths):
    all_preds = []

    for path in checkpoint_paths:
        model = RNN(
            input_size=test_loader.dataset[0][0].shape[1],
            rnn_units=48,   # We will override below
            fc_units=48
        ).to(device)

        model.load_state_dict(torch.load(path))
        model.eval()

        preds = []
        with torch.no_grad():
            for X, _ in test_loader:
                X = X.to(device)
                preds.append(model(X).cpu())

        all_preds.append(torch.cat(preds))

    all_preds = torch.stack(all_preds)  # shape: (num_models, N, 1)
    final_pred = all_preds.mean(dim=0)  # average over models

    return final_pred


In [ ]:
print("\nLoading test data...")
test_dataloader = load_test_data(test_path = "../../datasets_challenge/test",
                                 batchsize = 64,
                                 pipeline = pipeline, 
                                 load_anonymized_data=True)

In [ ]:
final_predictions = ensemble_predict(test_dataloader, checkpoint_paths)


In [ ]:
submission = pd.DataFrame({
    "index": np.arange(len(final_predictions)),
    "sideSlip": final_predictions.flatten().numpy()
})

submission.to_csv("ensemble_submission.csv", index=False)
print("Saved ensemble_submission.csv")
